# LeetCode 1455 — Check If a Word Occurs As a Prefix of Any Word in a Sentence bằng Python

**Link đề:** https://leetcode.com/problems/check-if-a-word-occurs-as-a-prefix-of-any-word-in-a-sentence/
**Độ khó:** Easy · **Chủ đề:** Two Pointers, String, String Matching

```python
class Solution:
    def isPrefixOfWord(self, sentence: str, searchWord: str) -> int:
        ...
```

### Mục tiêu học
1. Hiểu chính xác **prefix** (tiền tố) khác **substring** (chuỗi con bất kỳ) thế nào.
2. Dùng thành thạo `str.split()`, `enumerate(..., start=1)`, slicing `w[:m]`, `str.startswith`, `str.find`.
3. Đi qua 5 lời giải: so từng ký tự thủ công → slicing → `startswith` → tìm `" " + searchWord` trong câu → **một lượt hai con trỏ** không tạo list từ.
4. Nắm quy ước **chỉ số bắt đầu từ 1** (1-indexed) và giá trị mặc định `-1`.
5. Trace, chứng minh, Time/Space, lỗi hay gặp, biến thể (mọi vị trí khớp, suffix, đếm từ có tiền tố — *Counting Words With a Given Prefix*).

**Cách học:** đọc markdown → đoán output cell code kế tiếp → chạy → so sánh. Cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho `sentence` gồm các từ cách nhau bởi **đúng một dấu cách**, và `searchWord`. Kiểm tra `searchWord` có là **prefix** của từ nào trong câu không.

Trả về **chỉ số của từ** (đếm từ **1**) mà `searchWord` là prefix của nó. Nếu nhiều từ thỏa, trả chỉ số **nhỏ nhất**. Nếu không có, trả `-1`.

**Prefix** của chuỗi `s` là một đoạn **liên tiếp ở đầu** của `s`.

| Ví dụ | `sentence` | `searchWord` | Output |
|---|---|---|---:|
| 1 | `"i love eating burger"` | `"burg"` | 4 |
| 2 | `"this problem is an easy problem"` | `"pro"` | 2 |
| 3 | `"i am tired"` | `"you"` | -1 |

**Constraints**
- `1 <= sentence.length <= 100`
- `1 <= searchWord.length <= 10`
- `sentence` gồm chữ thường và dấu cách; `searchWord` gồm chữ thường.

Câu hỏi gợi mở:
- `"pro"` có trong `"improve"` không? Có là **prefix** của `"improve"` không?
- Ví dụ 2 có hai từ `"problem"` (vị trí 2 và 6) — trả cái nào?
- `searchWord` dài hơn từ, ví dụ `"burgers"` với từ `"burger"`, thì sao?

In [1]:
sentence = "i love eating burger"
words = sentence.split()
for idx, w in enumerate(words, start=1):
    print(idx, repr(w))

1 'i'
2 'love'
3 'eating'
4 'burger'


## 2. Nền tảng Python cần dùng

### 2.1. `str.split()` — tách câu thành list từ

- `s.split()` (không đối số) tách theo **mọi khoảng trắng** và bỏ chuỗi rỗng.
- `s.split(" ")` tách theo đúng một dấu cách; nếu có hai dấu cách liền nhau sẽ sinh chuỗi rỗng `''`.

Đề đảm bảo đúng một dấu cách nên hai cách như nhau, nhưng biết sự khác biệt giúp tránh lỗi ở bài khác.

In [2]:
print("i love  you".split())       # ['i', 'love', 'you']
print("i love  you".split(" "))    # ['i', 'love', '', 'you']
print("burger".split())            # ['burger'] — câu một từ

['i', 'love', 'you']
['i', 'love', '', 'you']
['burger']


### 2.2. Prefix và slicing `w[:m]`

`w[:m]` lấy `m` ký tự **đầu** của `w`. `searchWord` là prefix của `w` ⇔ `w[:len(searchWord)] == searchWord`.

Điều thú vị: nếu `m > len(w)`, slicing **không báo lỗi** mà trả cả `w` — khi đó so sánh tự động `False` vì độ dài khác nhau.

In [3]:
w = "burger"
print(w[:4], w[:4] == "burg")       # burg True
print(w[:3], w[:3] == "urg")        # bur False — 'urg' nằm trong nhưng không ở đầu
print(w[:10])                        # burger — m vượt độ dài: không lỗi
print(w[:7] == "burgers")           # False

burg True
bur False
burger
False


### 2.3. `str.startswith` — hàm có sẵn

`w.startswith(p)` trả `True` nếu `w` bắt đầu bằng `p`. Đây đúng là phép kiểm tra prefix, viết rõ ý nhất.

In [4]:
print("burger".startswith("burg"))     # True
print("improve".startswith("pro"))     # False
print("pro" in "improve")              # True — `in` kiểm tra SUBSTRING, không phải prefix!
print("burger".startswith(""))         # True — chuỗi rỗng là prefix của mọi chuỗi

True
False
True
True


Dòng thứ 3 là cái bẫy chính của bài: toán tử `in` với chuỗi kiểm tra **chuỗi con ở bất kỳ đâu**.

### 2.4. `enumerate(iterable, start=1)` — đếm từ 1

Đề yêu cầu chỉ số **1-indexed**. Thay vì `return i + 1`, ta cho `enumerate` bắt đầu đếm từ 1.

In [5]:
for i, w in enumerate(["a", "b", "c"]):
    print("mặc định:", i, w)
for i, w in enumerate(["a", "b", "c"], start=1):
    print("start=1 :", i, w)

mặc định: 0 a
mặc định: 1 b
mặc định: 2 c
start=1 : 1 a
start=1 : 2 b
start=1 : 3 c


### 2.5. `str.find` — vị trí xuất hiện đầu tiên

`s.find(t)` trả chỉ số **đầu tiên** mà `t` xuất hiện trong `s`, hoặc `-1` nếu không có. `s.count(" ", 0, k)` đếm số dấu cách trong `s[0:k]`.

In [6]:
s = " i love eating burger"
print(s.find(" burg"))        # 14
print(s.find(" xyz"))         # -1
print(s.count(" ", 0, 14))    # 3 dấu cách đứng trước vị trí 14

14
-1
3


## 3. Cách 1 — So từng ký tự thủ công (trực giác, không dùng hàm có sẵn)

Cách người mới tự nghĩ: tách câu thành từ; với mỗi từ, so từng ký tự của `searchWord` với ký tự tương ứng của từ. Nếu từ ngắn hơn `searchWord` thì chắc chắn không khớp.

In [7]:
def is_prefix_manual(word, p):
    if len(p) > len(word):
        return False
    for k in range(len(p)):
        if word[k] != p[k]:
            return False
    return True

def prefix_index_manual(sentence, searchWord):
    words = sentence.split()
    for i in range(len(words)):
        if is_prefix_manual(words[i], searchWord):
            return i + 1                   # 1-indexed
    return -1

print(prefix_index_manual("i love eating burger", "burg"))              # 4
print(prefix_index_manual("this problem is an easy problem", "pro"))    # 2
print(prefix_index_manual("i am tired", "you"))                         # -1

4
2
-1


### 3.1. Đánh giá cách 1

- Đúng, và đã là `O(n)` với `n = len(sentence)`: mỗi từ chỉ so tối đa `m = len(searchWord)` ký tự, tổng không vượt `n + m·(số từ)`.
- Nhưng nhiều dòng, dễ sai ở kiểm tra độ dài và `i + 1`.

Với bài này **không có nút thắt hiệu năng** thật sự (n ≤ 100). Các cách sau tập trung vào: (a) viết gọn, rõ ý; (b) tránh tạo list từ — tiết kiệm bộ nhớ, điều có ích khi câu rất dài.

## 4. Insight — "prefix của một từ" = "khớp ngay sau một dấu cách"

Một từ bắt đầu ở vị trí 0 hoặc **ngay sau một dấu cách**. Nếu ta thêm một dấu cách vào đầu câu: `" " + sentence`, thì **mọi** từ đều bắt đầu ngay sau một dấu cách. Khi đó:

> `searchWord` là prefix của một từ ⇔ `" " + searchWord` xuất hiện trong `" " + sentence`.

Và **số dấu cách đứng trước** vị trí khớp chính là **chỉ số 1-indexed** của từ đó (từ đầu tiên có 1 dấu cách trước nó — chính dấu cách ta thêm vào).

Kiểm tra tay trên ví dụ 2:

In [8]:
s = " " + "this problem is an easy problem"
pat = " " + "pro"
pos = s.find(pat)
print(repr(s))
print("vị trí khớp đầu tiên:", pos)
print("số dấu cách trong s[0:pos+1]:", s.count(" ", 0, pos + 1), "→ chỉ số từ")

' this problem is an easy problem'
vị trí khớp đầu tiên: 5
số dấu cách trong s[0:pos+1]: 2 → chỉ số từ


Đếm trong `s[0 : pos+1]` để **tính cả** dấu cách ở chính vị trí `pos` (dấu cách mở đầu từ cần tìm).

Ghi chú: `str.find` của CPython có độ phức tạp trung bình gần tuyến tính; với `m ≤ 10` có thể coi là `O(n · m)` tệ nhất.

## 5. Cách 2 — Slicing `w[:m] == searchWord`

In [9]:
def prefix_index_slice(sentence, searchWord):
    m = len(searchWord)
    for i, w in enumerate(sentence.split(), start=1):
        if w[:m] == searchWord:
            return i
    return -1

print(prefix_index_slice("i love eating burger", "burg"),
      prefix_index_slice("this problem is an easy problem", "pro"),
      prefix_index_slice("i am tired", "you"))

4 2 -1


## 6. Cách 3 (lời giải chính) — `split` + `startswith`

### 6.1. Mổ xẻ từng đoạn

**Đoạn 1 — tách từ:** `words = sentence.split()`.

**Đoạn 2 — duyệt theo thứ tự, đánh số từ 1:**
```python
for i, w in enumerate(words, start=1):
    if w.startswith(searchWord):
        return i          # trả NGAY → tự động là chỉ số nhỏ nhất
```

**Đoạn 3 — không thấy:** `return -1`.

Vì duyệt **từ trái sang phải** và `return` ngay ở từ khớp đầu tiên, ta luôn nhận chỉ số **nhỏ nhất** mà không cần so sánh thêm.

In [10]:
def prefix_index(sentence, searchWord):
    for i, w in enumerate(sentence.split(), start=1):
        if w.startswith(searchWord):
            return i
    return -1

print(prefix_index("i love eating burger", "burg"))
print(prefix_index("this problem is an easy problem", "pro"))
print(prefix_index("i am tired", "you"))

4
2
-1


## 7. TRACE từng bước — ví dụ 2 (`searchWord = "pro"`)

| i | từ `w` | `w[:3]` | `startswith("pro")`? | Hành động |
|---:|---|---|---|---|
| 1 | this | thi | không | tiếp |
| 2 | problem | pro | **có** | **return 2** |

Từ `problem` thứ hai (vị trí 6) **không bao giờ được xét** — đã dừng ở vị trí 2.

In [11]:
def trace_prefix_index(sentence, searchWord):
    m = len(searchWord)
    for i, w in enumerate(sentence.split(), start=1):
        ok = w.startswith(searchWord)
        print(f"i={i} w={w!r:12} w[:{m}]={w[:m]!r:8} khớp={ok}")
        if ok:
            print("→ return", i)
            return i
    print("→ không từ nào khớp, return -1")
    return -1

trace_prefix_index("this problem is an easy problem", "pro")

i=1 w='this'       w[:3]='thi'    khớp=False
i=2 w='problem'    w[:3]='pro'    khớp=True
→ return 2


2

### 7.1. Trace các nhánh đặc biệt: chuỗi con giữa từ, `searchWord` dài hơn từ

In [12]:
trace_prefix_index("i improve burger", "pro")
print()
trace_prefix_index("i love burger", "burgers")

i=1 w='i'          w[:3]='i'      khớp=False
i=2 w='improve'    w[:3]='imp'    khớp=False
i=3 w='burger'     w[:3]='bur'    khớp=False
→ không từ nào khớp, return -1

i=1 w='i'          w[:7]='i'      khớp=False
i=2 w='love'       w[:7]='love'   khớp=False
i=3 w='burger'     w[:7]='burger' khớp=False
→ không từ nào khớp, return -1


-1

- `"improve"` chứa `"pro"` ở giữa nhưng `w[:3] = "imp"` → không khớp. ✓
- `"burger"[:7] = "burger" != "burgers"` → không khớp, không lỗi. ✓

## 8. Cách 4 — Tìm `" " + searchWord` trong `" " + sentence`

Hiện thực hóa insight ở mục 4: một lần `find` + một lần `count`.

In [13]:
def prefix_index_find(sentence, searchWord):
    s = " " + sentence
    pos = s.find(" " + searchWord)
    if pos == -1:
        return -1
    return s.count(" ", 0, pos + 1)

print(prefix_index_find("i love eating burger", "burg"),
      prefix_index_find("this problem is an easy problem", "pro"),
      prefix_index_find("i am tired", "you"),
      prefix_index_find("i improve burger", "pro"))

4 2 -1 -1


Vì `find` trả vị trí **đầu tiên** nên ta cũng nhận đúng chỉ số **nhỏ nhất**.

## 9. Cách 5 — Một lượt hai con trỏ, không tạo list từ

Duyệt `sentence` bằng con trỏ `i`; biến `word_idx` đếm từ hiện tại; biến `at_start` cho biết `i` có phải đầu một từ không. Ở **đầu mỗi từ**, dùng con trỏ thứ hai `k` so với `searchWord`.

- Gặp dấu cách: `word_idx += 1`, đánh dấu ký tự sau là đầu từ.
- Ở đầu từ: so `sentence[i + k]` với `searchWord[k]` cho `k = 0..m-1` (dừng nếu gặp dấu cách / hết câu / khác ký tự).

In [14]:
def prefix_index_two_pointers(sentence, searchWord):
    n, m = len(sentence), len(searchWord)
    word_idx = 1
    at_start = True
    for i in range(n):
        if sentence[i] == " ":
            word_idx += 1
            at_start = True
            continue
        if at_start:
            k = 0
            while k < m and i + k < n and sentence[i + k] == searchWord[k]:
                k += 1
            if k == m:
                return word_idx
            at_start = False
    return -1

print(prefix_index_two_pointers("i love eating burger", "burg"),
      prefix_index_two_pointers("this problem is an easy problem", "pro"),
      prefix_index_two_pointers("i am tired", "you"),
      prefix_index_two_pointers("i improve burger", "pro"))

4 2 -1 -1


Vì `searchWord` không chứa dấu cách, điều kiện `sentence[i + k] == searchWord[k]` tự động dừng khi gặp dấu cách — không lo khớp tràn sang từ sau.

Đây là lý do LeetCode gắn tag **Two Pointers**: một con trỏ chạy trên câu, một con trỏ chạy trên `searchWord`.

## 10. Code nộp LeetCode (Python 3)

In [15]:
class Solution:
    def isPrefixOfWord(self, sentence: str, searchWord: str) -> int:
        for i, word in enumerate(sentence.split(), start=1):
            if word.startswith(searchWord):
                return i
        return -1

sol = Solution()
print(sol.isPrefixOfWord("i love eating burger", "burg"))             # 4
print(sol.isPrefixOfWord("this problem is an easy problem", "pro"))   # 2
print(sol.isPrefixOfWord("i am tired", "you"))                        # -1

4
2
-1


Bản thay thế (cách 4), cũng Accepted:

```python
class Solution:
    def isPrefixOfWord(self, sentence: str, searchWord: str) -> int:
        s = " " + sentence
        pos = s.find(" " + searchWord)
        return -1 if pos == -1 else s.count(" ", 0, pos + 1)
```

## 11. Vì sao thuật toán đúng?

**Cách 3:**
- `sentence.split()` cho đúng danh sách từ theo thứ tự (đề đảm bảo phân cách bằng một dấu cách).
- **Invariant:** khi xét từ thứ `i`, mọi từ `1..i-1` đều **không** có `searchWord` là prefix (nếu có, hàm đã `return` sớm hơn).
- Nếu trả `i` thì từ `i` có prefix `searchWord` và theo invariant nó là chỉ số **nhỏ nhất**. Nếu duyệt hết mà không trả → không từ nào khớp → `-1`. ✓

**Cách 4:** trong `s = " " + sentence`, mỗi dấu cách đứng ngay trước đúng một từ và ngược lại. `" " + p` xuất hiện tại `pos` ⇔ có từ bắt đầu ở `pos + 1` với prefix `p` (vì `p` không chứa dấu cách, phần khớp nằm gọn trong từ đó). Số dấu cách trong `s[0..pos]` = số từ bắt đầu tại hoặc trước `pos + 1` = chỉ số 1-indexed của từ đó. `find` trả vị trí nhỏ nhất ⇔ từ có chỉ số nhỏ nhất. ✓

## 12. Phân tích Time / Space

Gọi `n = len(sentence)`, `m = len(searchWord)`, `w` = số từ.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. So thủ công | `O(n + w·m)` ⊆ `O(n·m)` | `O(n)` (list từ) | dài dòng |
| 2. Slicing | `O(n + w·m)` | `O(n + m)` | tạo chuỗi `w[:m]` |
| 3. `startswith` | `O(n + w·m)` | `O(n)` | **khuyên dùng** |
| 4. `find` + `count` | `O(n·m)` tệ nhất | `O(n)` (chuỗi mới `" " + sentence`) | mẹo gọn |
| 5. Hai con trỏ | `O(n + w·m)` | `O(1)` | không tạo list |

Vì `m ≤ 10`, mọi cách đều thực chất tuyến tính theo `n`.

## 13. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

sol = Solution()
check("ví dụ 1", sol.isPrefixOfWord("i love eating burger", "burg"), 4)
check("ví dụ 2 (lấy chỉ số nhỏ nhất)", sol.isPrefixOfWord("this problem is an easy problem", "pro"), 2)
check("ví dụ 3", sol.isPrefixOfWord("i am tired", "you"), -1)
check("chuỗi con giữa từ không tính", sol.isPrefixOfWord("i improve", "pro"), -1)
check("searchWord dài hơn từ", sol.isPrefixOfWord("burger", "burgers"), -1)
check("searchWord bằng cả từ", sol.isPrefixOfWord("hello world", "world"), 2)
check("câu một từ", sol.isPrefixOfWord("a", "a"), 1)
check("khớp ở từ đầu tiên", sol.isPrefixOfWord("apple banana", "app"), 1)
check("khớp ở từ cuối", sol.isPrefixOfWord("x y zebra", "ze"), 3)
check("'ell' chỉ nằm giữa từ", sol.isPrefixOfWord("hellohello hellohellohello", "ell"), -1)

PASS  ví dụ 1: 4
PASS  ví dụ 2 (lấy chỉ số nhỏ nhất): 2
PASS  ví dụ 3: -1
PASS  chuỗi con giữa từ không tính: -1
PASS  searchWord dài hơn từ: -1
PASS  searchWord bằng cả từ: 2
PASS  câu một từ: 1
PASS  khớp ở từ đầu tiên: 1
PASS  khớp ở từ cuối: 3
PASS  'ell' chỉ nằm giữa từ: -1


### 13.1. Đối chiếu 5 cách — ngẫu nhiên có seed

Dùng bảng chữ nhỏ `"ab"` để có nhiều trường hợp khớp/không khớp chồng chéo.

In [17]:
import random
rng = random.Random(1455)
methods = [prefix_index_manual, prefix_index_slice, prefix_index, prefix_index_find,
           prefix_index_two_pointers, Solution().isPrefixOfWord]
for _ in range(2000):
    words = ["".join(rng.choice("ab") for _ in range(rng.randint(1, 5))) for _ in range(rng.randint(1, 8))]
    sentence = " ".join(words)
    search = "".join(rng.choice("ab") for _ in range(rng.randint(1, 4)))
    results = {f(sentence, search) for f in methods}
    assert len(results) == 1, (sentence, search, results)
print("2000 bộ ngẫu nhiên: 6 hàm cho cùng kết quả")

2000 bộ ngẫu nhiên: 6 hàm cho cùng kết quả


## 14. Lỗi người mới hay gặp

### Lỗi 1 — Dùng `in` (substring) thay vì prefix

In [18]:
def wrong_in(sentence, searchWord):
    for i, w in enumerate(sentence.split(), start=1):
        if searchWord in w:
            return i
    return -1

print("Sai:", wrong_in("i improve problem", "pro"), "| Đúng:", prefix_index("i improve problem", "pro"))

Sai: 2 | Đúng: 3


**Sửa:** `w.startswith(searchWord)`.

### Lỗi 2 — Trả chỉ số 0-indexed

In [19]:
def wrong_zero_index(sentence, searchWord):
    for i, w in enumerate(sentence.split()):
        if w.startswith(searchWord):
            return i
    return -1

print("Sai:", wrong_zero_index("i love eating burger", "burg"), "| Đúng:", prefix_index("i love eating burger", "burg"))

Sai: 3 | Đúng: 4


**Sửa:** `enumerate(..., start=1)` hoặc `return i + 1`.

### Lỗi 3 — `find` trên câu gốc, không thêm dấu cách

`sentence.find(searchWord)` tìm cả chỗ **giữa từ**, và chỉ số trả về là **vị trí ký tự**, không phải chỉ số từ.

In [20]:
def wrong_find(sentence, searchWord):
    pos = sentence.find(searchWord)
    return -1 if pos == -1 else sentence.count(" ", 0, pos) + 1

print("Sai:", wrong_find("i improve problem", "pro"), "| Đúng:", prefix_index_find("i improve problem", "pro"))

Sai: 2 | Đúng: 3


### Lỗi 4 — Trả chỉ số **lớn nhất** (hoặc chỉ số cuối) do không `return` sớm

In [21]:
def wrong_last(sentence, searchWord):
    ans = -1
    for i, w in enumerate(sentence.split(), start=1):
        if w.startswith(searchWord):
            ans = i                  # ghi đè → giữ lần khớp CUỐI
    return ans

print("Sai:", wrong_last("this problem is an easy problem", "pro"), "| Đúng:", prefix_index("this problem is an easy problem", "pro"))

Sai: 6 | Đúng: 2


**Sửa:** `return i` ngay lần khớp đầu, hoặc chỉ gán khi `ans == -1`.

### Lỗi 5 — So từng ký tự mà quên kiểm tra độ dài → `IndexError`

In [22]:
def wrong_manual(word, p):
    for k in range(len(p)):
        if word[k] != p[k]:
            return False
    return True

try:
    wrong_manual("burger", "burgers")
except IndexError as e:
    print("Lỗi:", e, "→ cần kiểm tra len(p) > len(word) trước")

Lỗi: string index out of range → cần kiểm tra len(p) > len(word) trước


## 15. Mẹo nhận dạng pattern

- "Prefix / bắt đầu bằng" → `startswith` hoặc so `w[:m]`; **đừng** dùng `in`.
- "Trả vị trí nhỏ nhất / đầu tiên thỏa" → duyệt trái sang phải và `return` ngay.
- "Chỉ số 1-indexed" → `enumerate(..., start=1)`.
- Mẹo **thêm ký tự phân cách vào đầu** (`" " + s`) biến bài "đầu từ" thành bài "tìm chuỗi con" — dùng lại được với dấu phẩy, `/` trong đường dẫn, v.v.
- Khi có nhiều truy vấn prefix trên cùng tập từ → cấu trúc **Trie** (cây tiền tố).
- Bài cùng họ: *Counting Words With a Given Prefix* (2185), *Longest Common Prefix* (14), *Count Prefixes of a Given String* (2255), *Implement Trie* (208).

## 16. Biến thể

### 16.1. Trả **mọi** chỉ số từ có prefix

In [23]:
def all_prefix_indices(sentence, searchWord):
    return [i for i, w in enumerate(sentence.split(), start=1) if w.startswith(searchWord)]

print(all_prefix_indices("this problem is an easy problem", "pro"))   # [2, 6]

[2, 6]


### 16.2. Suffix thay vì prefix — `endswith`

In [24]:
def suffix_index(sentence, searchWord):
    for i, w in enumerate(sentence.split(), start=1):
        if w.endswith(searchWord):
            return i
    return -1

print(suffix_index("i love eating burger", "ing"))   # 3

3


### 16.3. Counting Words With a Given Prefix (LeetCode 2185)

In [25]:
def prefix_count(words, pref):
    return sum(w.startswith(pref) for w in words)

print(prefix_count(["pay", "attention", "practice", "attend"], "at"))   # 2

2


## 17. Bài tập luyện tập (tăng dần)

**Bài 1 — Dự đoán output.**
```python
w = "eating"
print(w[:3], w[:20], w.startswith("eat"), "tin" in w, w.startswith("tin"))
```

**Bài 2 — Tính tay cách 4.** `sentence = "a bb abc ab"`, `searchWord = "ab"`. Viết `s = " " + sentence`, tìm `pos = s.find(" ab")`, đếm dấu cách trong `s[0:pos+1]`.

Bài 3–6 có cell TODO.

### Bài 3 — TODO: tự viết lời giải `startswith`

In [26]:
def exercise_prefix_index(sentence, searchWord):
    # TODO: split + enumerate(start=1) + startswith, return sớm; cuối cùng -1
    pass

# assert exercise_prefix_index("i love eating burger", "burg") == 4
# assert exercise_prefix_index("i am tired", "you") == -1

### Bài 4 — TODO: viết lại không dùng `split` và không dùng `startswith`

In [27]:
def exercise_prefix_no_split(sentence, searchWord):
    # TODO: duyệt từng ký tự, đếm số từ, ở đầu từ so tiếp searchWord bằng con trỏ thứ hai
    pass

# assert exercise_prefix_no_split("this problem is an easy problem", "pro") == 2
# assert exercise_prefix_no_split("i improve", "pro") == -1

### Bài 5 — TODO: Longest Common Prefix (LeetCode 14)
Trả tiền tố chung dài nhất của mọi chuỗi trong `strs` (có thể là `""`).

In [28]:
def exercise_longest_common_prefix(strs):
    # TODO: so từng cột ký tự k của mọi chuỗi; dừng khi có chuỗi hết hoặc ký tự khác
    pass

# assert exercise_longest_common_prefix(["flower","flow","flight"]) == "fl"
# assert exercise_longest_common_prefix(["dog","racecar","car"]) == ""

### Bài 6 — Mini challenge: nhiều truy vấn
Cho `sentence` và list `queries`. Trả list đáp án (như bài gốc) cho từng truy vấn, sao cho mỗi truy vấn chỉ tốn `O(m)` sau tiền xử lý.

In [29]:
def exercise_many_queries(sentence, queries):
    # TODO: tiền xử lý dict first[prefix] = chỉ số từ nhỏ nhất có prefix đó
    #       (mỗi từ có tối đa 10 prefix cần quan tâm vì len(searchWord) <= 10)
    pass

# assert exercise_many_queries("this problem is an easy problem", ["pro", "e", "z", "is"]) == [2, 5, -1, 3]

## 18. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 3 | Duyệt từ theo thứ tự, dừng ở lần khớp đầu. | `for i, w in enumerate(words, start=1)` | `if w.startswith(searchWord): return i` |
| 4 | Đầu từ = vị trí 0 hoặc ngay sau dấu cách. | biến `word_idx`, `at_start`, con trỏ `k`. | xem cách 5 ở mục 9 nhưng tự viết lại |
| 5 | Prefix chung không dài hơn chuỗi ngắn nhất. | vòng ngoài theo cột `k`, vòng trong theo chuỗi. | `if k == len(s) or s[k] != strs[0][k]: return strs[0][:k]` |
| 6 | Tiền xử lý mọi prefix (≤ 10) của mỗi từ. | `dict` + `setdefault` để giữ chỉ số nhỏ nhất. | `for L in range(1, min(10, len(w)) + 1): first.setdefault(w[:L], i)` |

## 19. Đáp án tham khảo (chỉ xem sau khi tự làm)

**Bài 1:** `eat eating True True False`.
**Bài 2:** `s = " a bb abc ab"`, `pos = 5` (dấu cách trước `abc`), dấu cách trong `s[0:6]` là 3 → đáp án **3**.

In [30]:
def answer_prefix_index(sentence, searchWord):
    for i, w in enumerate(sentence.split(), start=1):
        if w.startswith(searchWord):
            return i
    return -1

def answer_prefix_no_split(sentence, searchWord):
    n, m = len(sentence), len(searchWord)
    word_idx, i = 1, 0
    while i < n:
        k = 0
        while k < m and i + k < n and sentence[i + k] == searchWord[k]:
            k += 1
        if k == m:
            return word_idx
        while i < n and sentence[i] != " ":   # nhảy tới hết từ hiện tại
            i += 1
        i += 1                                 # bỏ qua dấu cách
        word_idx += 1
    return -1

def answer_longest_common_prefix(strs):
    first = strs[0]
    for k in range(len(first)):
        for s in strs[1:]:
            if k == len(s) or s[k] != first[k]:
                return first[:k]
    return first

def answer_many_queries(sentence, queries):
    first = {}
    for i, w in enumerate(sentence.split(), start=1):
        for L in range(1, min(10, len(w)) + 1):
            first.setdefault(w[:L], i)       # chỉ gán nếu chưa có → giữ chỉ số nhỏ nhất
    return [first.get(q, -1) for q in queries]

assert answer_prefix_index("i love eating burger", "burg") == 4
assert answer_prefix_index("i am tired", "you") == -1
assert answer_prefix_no_split("this problem is an easy problem", "pro") == 2
assert answer_prefix_no_split("i improve", "pro") == -1
assert answer_longest_common_prefix(["flower","flow","flight"]) == "fl"
assert answer_longest_common_prefix(["dog","racecar","car"]) == ""
assert answer_many_queries("this problem is an easy problem", ["pro", "e", "z", "is"]) == [2, 5, -1, 3]
for _ in range(500):
    words = ["".join(rng.choice("ab") for _ in range(rng.randint(1, 5))) for _ in range(rng.randint(1, 8))]
    sen = " ".join(words)
    q = "".join(rng.choice("ab") for _ in range(rng.randint(1, 4)))
    assert answer_prefix_no_split(sen, q) == answer_many_queries(sen, [q])[0] == prefix_index(sen, q)
w = "eating"
print("Bài 1:", w[:3], w[:20], w.startswith("eat"), "tin" in w, w.startswith("tin"))
s = " " + "a bb abc ab"; pos = s.find(" ab")
print("Bài 2: pos =", pos, "→", s.count(" ", 0, pos + 1))
print("Đáp án bài 3–6: PASS")

Bài 1: eat eating True True False
Bài 2: pos = 5 → 3
Đáp án bài 3–6: PASS


## 20. Template nhớ nhanh + checklist submit

```python
for i, w in enumerate(sentence.split(), start=1):   # 1-indexed
    if w.startswith(searchWord):                     # PREFIX, không phải `in`
        return i                                     # đầu tiên = nhỏ nhất
return -1
```

**Checklist trước khi nộp**
- [ ] Kiểm tra **prefix** (`startswith` / `w[:m]`), không phải substring (`in`).
- [ ] Trả chỉ số **1-indexed**.
- [ ] Trả lần khớp **đầu tiên** (return sớm).
- [ ] `searchWord` dài hơn từ không gây lỗi.
- [ ] Không khớp → `-1`.
- [ ] Method tên `isPrefixOfWord`, trả về `int`.